In [ ]:
!nvidia-smi -L
!nvcc --version


In [ ]:
%%writefile init_sin.cu
// init_sin.cu
// Lab task: initialise arr[i] = sin((i%360) * Pi / 180) on the GPU for N=1e9,
// copy to host, compute mean |error|. Compare sin / sinf / __sinf
// and float vs double. Report kernel timings, mean/min/max, bandwidth,
// and the error distribution across the 360 angles.
//
// Build (Colab, Tesla T4 or similar):
//   nvcc -O3 -std=c++17 -arch=native -Xcompiler -fopenmp init_sin.cu -o init_sin
// NOTE: --use_fast_math is intentionally NOT passed. It would silently rewrite
// sinf -> __sinf everywhere, hiding the comparison we are supposed to make.
//
// Run:
//   ./init_sin [N=1000000000] [reps=5] [out_prefix=results]
// Produces:
//   <prefix>_timings.csv       — one row per (type, func)
//   <prefix>_err_per_angle.csv — one column per (type, func), 360 rows

#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cmath>
#include <cstdint>
#include <algorithm>
#include <limits>
#include <string>
#include <vector>

#include <cuda_runtime.h>
#include <omp.h>

// Nearest double to pi. We reuse the SAME constant on host and device so the
// reference and the GPU "double" path can agree to within a ULP.
static const double PI_D = 3.14159265358979323846;

#define CUDA_CHECK(call) do {                                               \
    cudaError_t _e = (call);                                                \
    if (_e != cudaSuccess) {                                                \
        std::fprintf(stderr, "CUDA error at %s:%d (%s): %s\n",              \
                     __FILE__, __LINE__, #call, cudaGetErrorString(_e));    \
        std::exit(1);                                                       \
    }                                                                       \
} while (0)

// ---------------------------------------------------------------------------
// Kernels
//
// Each kernel does one write per element. The grid-stride loop keeps the
// launch config independent of N.
//
// For the FLOAT array we deliberately keep the ARGUMENT computed in double
// (double k * PI / 180) so the error we measure comes from the function and
// the storage cast, not from losing precision in the angle computation.
//
// For the DOUBLE array we do the same thing. For "sinf" and "__sinf" with a
// double destination we must still cast the argument down to float because
// those functions are intrinsically single-precision — that's the whole point
// of the experiment: double storage cannot rescue a single-precision function.
// ---------------------------------------------------------------------------

__global__ void k_float_sin(float* __restrict__ arr, long long N) {
    long long tid    = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long stride = (long long)blockDim.x * gridDim.x;
    for (long long i = tid; i < N; i += stride) {
        int    k     = (int)(i % 360);
        double angle = (double)k * PI_D / 180.0;        // argument in double
        arr[i]       = (float)sin(angle);               // double sin, cast at store
    }
}

__global__ void k_float_sinf(float* __restrict__ arr, long long N) {
    long long tid    = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long stride = (long long)blockDim.x * gridDim.x;
    for (long long i = tid; i < N; i += stride) {
        int   k     = (int)(i % 360);
        float angle = (float)((double)k * PI_D / 180.0); // computed in double, rounded to float
        arr[i]      = sinf(angle);                       // single-precision libm sin
    }
}

__global__ void k_float_fastsinf(float* __restrict__ arr, long long N) {
    long long tid    = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long stride = (long long)blockDim.x * gridDim.x;
    for (long long i = tid; i < N; i += stride) {
        int   k     = (int)(i % 360);
        float angle = (float)((double)k * PI_D / 180.0);
        arr[i]      = __sinf(angle);                     // hardware MUFU intrinsic
    }
}

__global__ void k_double_sin(double* __restrict__ arr, long long N) {
    long long tid    = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long stride = (long long)blockDim.x * gridDim.x;
    for (long long i = tid; i < N; i += stride) {
        int    k     = (int)(i % 360);
        double angle = (double)k * PI_D / 180.0;
        arr[i]       = sin(angle);                       // full double path
    }
}

__global__ void k_double_sinf(double* __restrict__ arr, long long N) {
    long long tid    = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long stride = (long long)blockDim.x * gridDim.x;
    for (long long i = tid; i < N; i += stride) {
        int   k     = (int)(i % 360);
        float angle = (float)((double)k * PI_D / 180.0);
        arr[i]      = (double)sinf(angle);               // single sin, widened for storage
    }
}

__global__ void k_double_fastsinf(double* __restrict__ arr, long long N) {
    long long tid    = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long stride = (long long)blockDim.x * gridDim.x;
    for (long long i = tid; i < N; i += stride) {
        int   k     = (int)(i % 360);
        float angle = (float)((double)k * PI_D / 180.0);
        arr[i]      = (double)__sinf(angle);
    }
}

// ---------------------------------------------------------------------------
// Timing helper. Warm-up launch is NOT timed because the first launch pays
// for JIT compile, PTX load, resource allocation, and clock ramp-up — those
// are startup costs, not steady-state cost.
// ---------------------------------------------------------------------------

struct Timing { double mean_ms, min_ms, max_ms; };

template <typename KFn>
Timing time_kernel(KFn launch, int reps) {
    cudaEvent_t s, e;
    CUDA_CHECK(cudaEventCreate(&s));
    CUDA_CHECK(cudaEventCreate(&e));

    launch();                                     // warm-up
    CUDA_CHECK(cudaDeviceSynchronize());

    double sum = 0.0;
    double mn  = std::numeric_limits<double>::infinity();
    double mx  = 0.0;
    for (int r = 0; r < reps; ++r) {
        CUDA_CHECK(cudaEventRecord(s));
        launch();
        CUDA_CHECK(cudaEventRecord(e));
        CUDA_CHECK(cudaEventSynchronize(e));
        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, s, e));
        sum += ms;
        if (ms < mn) mn = ms;
        if (ms > mx) mx = ms;
    }
    CUDA_CHECK(cudaEventDestroy(s));
    CUDA_CHECK(cudaEventDestroy(e));
    return { sum / reps, mn, mx };
}

// ---------------------------------------------------------------------------
// Host-side error accumulation. Stream the array from device to host in
// chunks so we don't need the whole 4 GB / 8 GB in host memory at once.
// Every per-element error is up-promoted to double, and the SUM is in
// double. A float accumulator would be useless: adding ~1e9 terms of size
// ~1e-7 to a running sum that eventually reaches ~0.1–100 loses the small
// terms once the sum passes the float precision threshold.
// ---------------------------------------------------------------------------

struct ErrAccum {
    double                 sum_abs_err = 0.0;
    double                 max_abs_err = 0.0;
    std::vector<double>    per_angle_sum = std::vector<double>(360, 0.0);
    std::vector<long long> per_angle_cnt = std::vector<long long>(360, 0);
};

template <typename T>
void accumulate_errors_chunk(const T* host_chunk, long long chunk_n,
                             long long global_start, ErrAccum& acc) {
    const int nthr = omp_get_max_threads();
    std::vector<double>              psum(nthr, 0.0);
    std::vector<double>              pmax(nthr, 0.0);
    std::vector<std::vector<double>> pang(nthr, std::vector<double>(360, 0.0));
    std::vector<std::vector<long long>> pcnt(nthr, std::vector<long long>(360, 0));

    #pragma omp parallel
    {
        const int tid = omp_get_thread_num();
        double s = 0.0, m = 0.0;
        auto& as = pang[tid];
        auto& ac = pcnt[tid];
        #pragma omp for schedule(static)
        for (long long i = 0; i < chunk_n; ++i) {
            long long gi    = global_start + i;
            int       k     = (int)(gi % 360);
            double    angle = (double)k * PI_D / 180.0;
            double    ref   = sin(angle);                // host libm double sin
            double    err   = std::fabs(ref - (double)host_chunk[i]);
            s += err;
            if (err > m) m = err;
            as[k] += err;
            ac[k] += 1;
        }
        psum[tid] = s;
        pmax[tid] = m;
    }

    for (int t = 0; t < nthr; ++t) {
        acc.sum_abs_err += psum[t];
        if (pmax[t] > acc.max_abs_err) acc.max_abs_err = pmax[t];
        for (int k = 0; k < 360; ++k) {
            acc.per_angle_sum[k] += pang[t][k];
            acc.per_angle_cnt[k] += pcnt[t][k];
        }
    }
}

// ---------------------------------------------------------------------------
// One experiment. Returns timings + error summary + per-angle mean error.
// ---------------------------------------------------------------------------

struct Row {
    std::string         type;
    std::string         func;
    Timing              t;
    double              mean_err;
    double              max_abs_err;
    double              bandwidth_gb_s;
    std::vector<double> err_per_angle;
};

template <typename T, typename KFn>
Row run_variant(const std::string& type_name,
                const std::string& func_name,
                T* d_arr, long long N, int reps,
                KFn kernel_launch) {
    Row r;
    r.type = type_name;
    r.func = func_name;

    r.t = time_kernel(kernel_launch, reps);

    // Effective write bandwidth. The kernel reads nothing from device memory
    // (angles are computed from the thread index) and writes exactly one
    // element per i. So the only device-memory traffic is the write.
    const double bytes = (double)N * sizeof(T);
    r.bandwidth_gb_s = (bytes / 1.0e9) / (r.t.mean_ms / 1.0e3);

    // One final launch so the array holds its last-run contents, then stream
    // it back to host in manageable chunks for the error check.
    kernel_launch();
    CUDA_CHECK(cudaDeviceSynchronize());

    const long long CHUNK = 1LL << 26;       // 64 Mi elements: 256 MB float / 512 MB double
    std::vector<T> host(CHUNK);
    ErrAccum acc;
    for (long long off = 0; off < N; off += CHUNK) {
        long long n = std::min<long long>(CHUNK, N - off);
        CUDA_CHECK(cudaMemcpy(host.data(), d_arr + off,
                              n * sizeof(T), cudaMemcpyDeviceToHost));
        accumulate_errors_chunk<T>(host.data(), n, off, acc);
    }
    r.mean_err    = acc.sum_abs_err / (double)N;
    r.max_abs_err = acc.max_abs_err;
    r.err_per_angle.resize(360);
    for (int k = 0; k < 360; ++k) {
        r.err_per_angle[k] = acc.per_angle_cnt[k]
            ? acc.per_angle_sum[k] / (double)acc.per_angle_cnt[k]
            : 0.0;
    }
    return r;
}

// ---------------------------------------------------------------------------
// Main
// ---------------------------------------------------------------------------

int main(int argc, char** argv) {
    long long   N      = (argc > 1) ? std::atoll(argv[1]) : 1'000'000'000LL;
    int         reps   = (argc > 2) ? std::atoi(argv[2]) : 5;
    std::string prefix = (argc > 3) ? argv[3] : "results";

    int dev = 0;
    CUDA_CHECK(cudaGetDevice(&dev));
    cudaDeviceProp prop{};
    CUDA_CHECK(cudaGetDeviceProperties(&prop, dev));
    int rt = 0, dr = 0;
    cudaRuntimeGetVersion(&rt);
    cudaDriverGetVersion(&dr);
    size_t free_b = 0, total_b = 0;
    CUDA_CHECK(cudaMemGetInfo(&free_b, &total_b));

    std::printf("GPU: %s | SMs: %d | compute %d.%d\n",
                prop.name, prop.multiProcessorCount, prop.major, prop.minor);
    std::printf("CUDA runtime: %d.%d, driver: %d.%d\n",
                rt / 1000, (rt % 1000) / 10, dr / 1000, (dr % 1000) / 10);
    std::printf("Device memory: %.2f GB total, %.2f GB free\n",
                total_b / 1.0e9, free_b / 1.0e9);
    std::printf("N = %lld, reps = %d\n", N, reps);

    const int       block  = 256;
    const long long needed = (N + block - 1) / block;
    const int       grid   = (int)std::min<long long>(needed,
                                 (long long)prop.multiProcessorCount * 32);

    std::vector<Row> rows;

    // ---------- float experiments ----------
    {
        size_t bytes = (size_t)N * sizeof(float);
        if (bytes + (1ULL << 28) > free_b) {
            std::printf("[skip] float array (%.2f GB) does not fit with headroom.\n",
                        bytes / 1.0e9);
        } else {
            float* d = nullptr;
            CUDA_CHECK(cudaMalloc(&d, bytes));
            auto a = [&] { k_float_sin      <<<grid, block>>>(d, N); };
            auto b = [&] { k_float_sinf     <<<grid, block>>>(d, N); };
            auto c = [&] { k_float_fastsinf <<<grid, block>>>(d, N); };
            rows.push_back(run_variant<float>("float", "sin",    d, N, reps, a));
            rows.push_back(run_variant<float>("float", "sinf",   d, N, reps, b));
            rows.push_back(run_variant<float>("float", "__sinf", d, N, reps, c));
            CUDA_CHECK(cudaFree(d));
        }
    }

    CUDA_CHECK(cudaMemGetInfo(&free_b, &total_b));

    // ---------- double experiments ----------
    {
        size_t bytes = (size_t)N * sizeof(double);
        if (bytes + (1ULL << 28) > free_b) {
            std::printf("[skip] double array (%.2f GB) does not fit with headroom.\n",
                        bytes / 1.0e9);
        } else {
            double* d = nullptr;
            CUDA_CHECK(cudaMalloc(&d, bytes));
            auto a = [&] { k_double_sin      <<<grid, block>>>(d, N); };
            auto b = [&] { k_double_sinf     <<<grid, block>>>(d, N); };
            auto c = [&] { k_double_fastsinf <<<grid, block>>>(d, N); };
            rows.push_back(run_variant<double>("double", "sin",    d, N, reps, a));
            rows.push_back(run_variant<double>("double", "sinf",   d, N, reps, b));
            rows.push_back(run_variant<double>("double", "__sinf", d, N, reps, c));
            CUDA_CHECK(cudaFree(d));
        }
    }

    // ---------- CSV output ----------
    std::string ft = prefix + "_timings.csv";
    FILE* f = std::fopen(ft.c_str(), "w");
    std::fprintf(f, "type,func,mean_ms,min_ms,max_ms,mean_err,max_abs_err,bandwidth_GBs\n");
    for (auto& r : rows) {
        std::fprintf(f, "%s,%s,%.6f,%.6f,%.6f,%.6e,%.6e,%.3f\n",
                     r.type.c_str(), r.func.c_str(),
                     r.t.mean_ms, r.t.min_ms, r.t.max_ms,
                     r.mean_err, r.max_abs_err, r.bandwidth_gb_s);
    }
    std::fclose(f);

    std::string fa = prefix + "_err_per_angle.csv";
    f = std::fopen(fa.c_str(), "w");
    std::fprintf(f, "angle_deg");
    for (auto& r : rows) std::fprintf(f, ",%s_%s", r.type.c_str(), r.func.c_str());
    std::fprintf(f, "\n");
    for (int k = 0; k < 360; ++k) {
        std::fprintf(f, "%d", k);
        for (auto& r : rows) std::fprintf(f, ",%.6e", r.err_per_angle[k]);
        std::fprintf(f, "\n");
    }
    std::fclose(f);

    std::printf("\n%-7s %-7s %12s %12s %12s %14s %14s %12s\n",
                "type", "func", "mean_ms", "min_ms", "max_ms",
                "mean_err", "max_abs_err", "GB/s");
    for (auto& r : rows) {
        std::printf("%-7s %-7s %12.3f %12.3f %12.3f %14.3e %14.3e %12.1f\n",
                    r.type.c_str(), r.func.c_str(),
                    r.t.mean_ms, r.t.min_ms, r.t.max_ms,
                    r.mean_err, r.max_abs_err, r.bandwidth_gb_s);
    }
    std::printf("\nWrote %s and %s\n", ft.c_str(), fa.c_str());
    return 0;
}


In [ ]:
!nvcc -O3 -std=c++17 -arch=native -Xcompiler -fopenmp init_sin.cu -o init_sin


In [ ]:
import time
t0 = time.time()
!./init_sin 1000000000 5 results
print(f"\nTotal wall clock: {time.time()-t0:.1f} s")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df  = pd.read_csv('results_timings.csv')
dfa = pd.read_csv('results_err_per_angle.csv')

# A tiny floor so log-scale plots don't collapse at exact zeros.
EPS = 1e-20

# Consistent colouring across all plots so float vs double is readable at a glance.
TYPE_COLOR = {'float': '#1f77b4', 'double': '#d62728'}
FUNC_HATCH = {'sin': '', 'sinf': '//', '__sinf': 'xx'}

def xticklabels(df):
    return [f"{t}\n{f}" for t, f in zip(df['type'], df['func'])]

df


In [ ]:
x = np.arange(len(df))
y = np.maximum(df['mean_err'].values, EPS)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
bars = ax.bar(x, y,
              color=[TYPE_COLOR[t] for t in df['type']],
              edgecolor='black', linewidth=0.6)
for bar, func in zip(bars, df['func']):
    bar.set_hatch(FUNC_HATCH[func])

ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels(xticklabels(df))
ax.set_ylabel('mean |ref - arr[i]|')
ax.set_title('Mean absolute error per variant (log scale)')
ax.grid(True, axis='y', which='both', alpha=0.3)

# Annotate each bar with its value for the written-up explanation.
for xi, yi, raw in zip(x, y, df['mean_err']):
    ax.text(xi, yi * 1.4, f"{raw:.2e}", ha='center', va='bottom', fontsize=8)

fig.tight_layout()
fig.savefig('graph1_mean_error.png', dpi=150)
plt.show()


In [ ]:
x = np.arange(len(df))
mean = df['mean_ms'].values
mn   = df['min_ms'].values
mx   = df['max_ms'].values
yerr = np.vstack([mean - mn, mx - mean])

fig, ax = plt.subplots(figsize=(8.5, 4.8))
bars = ax.bar(x, mean, yerr=yerr, capsize=4,
              color=[TYPE_COLOR[t] for t in df['type']],
              edgecolor='black', linewidth=0.6)
for bar, func in zip(bars, df['func']):
    bar.set_hatch(FUNC_HATCH[func])

ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels(xticklabels(df))
ax.set_ylabel('kernel time (ms, log scale)')
ax.set_title('Kernel time per variant — mean with [min, max] whiskers')
ax.grid(True, axis='y', which='both', alpha=0.3)

for xi, m in zip(x, mean):
    ax.text(xi, m * 1.1, f"{m:.1f}", ha='center', va='bottom', fontsize=8)

fig.tight_layout()
fig.savefig('graph2_kernel_time.png', dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))

handles, labels = [], []
MARKER = {'sin': 'o', 'sinf': 's', '__sinf': '^'}
for _, row in df.iterrows():
    xv = row['mean_ms']
    yv = max(row['mean_err'], EPS)
    h = ax.scatter(xv, yv, s=120,
                   color=TYPE_COLOR[row['type']],
                   marker=MARKER[row['func']],
                   edgecolor='black', linewidth=0.6)
    handles.append(h)
    # labels starting with '_' are hidden by matplotlib by default — pass them
    # through explicit handles/labels instead of relying on auto-legend.
    labels.append(f"{row['type']} / {row['func']}")
    ax.annotate(f"{row['type']}/{row['func']}", (xv, yv),
                textcoords='offset points', xytext=(7, 5), fontsize=8)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('kernel time (ms)')
ax.set_ylabel('mean |error|')
ax.set_title('Speed vs accuracy — lower-left is better')
ax.grid(True, which='both', alpha=0.3)
ax.legend(handles=handles, labels=labels, loc='best', fontsize=8)

fig.tight_layout()
fig.savefig('graph3_tradeoff.png', dpi=150)
plt.show()


In [ ]:
x = np.arange(len(df))
bw = df['bandwidth_GBs'].values

fig, ax = plt.subplots(figsize=(8.5, 4.8))
bars = ax.bar(x, bw,
              color=[TYPE_COLOR[t] for t in df['type']],
              edgecolor='black', linewidth=0.6)
for bar, func in zip(bars, df['func']):
    bar.set_hatch(FUNC_HATCH[func])

ax.set_xticks(x)
ax.set_xticklabels(xticklabels(df))
ax.set_ylabel('effective write bandwidth (GB/s)')
ax.set_title('Achieved write bandwidth per variant')
ax.grid(True, axis='y', alpha=0.3)

# Reference line for T4 peak (ignore if your GPU is different; adjust the label).
ax.axhline(320, linestyle='--', color='gray', linewidth=1)
ax.text(len(df) - 0.6, 325, 'T4 peak ≈ 320 GB/s', fontsize=8, color='gray')

for xi, v in zip(x, bw):
    ax.text(xi, v + 5, f"{v:.0f}", ha='center', va='bottom', fontsize=8)

fig.tight_layout()
fig.savefig('graph4_bandwidth.png', dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))

handles, labels = [], []
STYLE = {'sin': ('-', 1.6), 'sinf': ('--', 1.4), '__sinf': (':', 1.8)}
for col in dfa.columns[1:]:
    if not col.startswith('float_'):
        continue
    func = col.split('_', 1)[1]
    ls, lw = STYLE.get(func, ('-', 1.3))
    y = np.maximum(dfa[col].values, EPS)
    (line,) = ax.plot(dfa['angle_deg'], y, ls, lw=lw,
                      color=TYPE_COLOR['float'] if func == 'sin' else
                            ('#2ca02c' if func == 'sinf' else '#9467bd'))
    handles.append(line)
    labels.append(f"float / {func}")

ax.set_yscale('log')
ax.set_xlabel('angle (degrees)')
ax.set_ylabel('mean |error| at that angle')
ax.set_title('Error per angle — float array')
ax.set_xticks(range(0, 361, 45))
ax.grid(True, which='both', alpha=0.3)
# Explicit handles/labels so the '__sinf' label (leading underscores) is NOT dropped.
ax.legend(handles=handles, labels=labels, loc='best', fontsize=8)

fig.tight_layout()
fig.savefig('graph5_err_per_angle_float.png', dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))

handles, labels = [], []
for col in dfa.columns[1:]:
    if not col.startswith('double_'):
        continue
    func = col.split('_', 1)[1]
    ls, lw = STYLE.get(func, ('-', 1.3))
    y = np.maximum(dfa[col].values, EPS)
    (line,) = ax.plot(dfa['angle_deg'], y, ls, lw=lw,
                      color=TYPE_COLOR['double'] if func == 'sin' else
                            ('#8c564b' if func == 'sinf' else '#e377c2'))
    handles.append(line)
    labels.append(f"double / {func}")

ax.set_yscale('log')
ax.set_xlabel('angle (degrees)')
ax.set_ylabel('mean |error| at that angle')
ax.set_title('Error per angle — double array')
ax.set_xticks(range(0, 361, 45))
ax.grid(True, which='both', alpha=0.3)
ax.legend(handles=handles, labels=labels, loc='best', fontsize=8)

fig.tight_layout()
fig.savefig('graph6_err_per_angle_double.png', dpi=150)
plt.show()


In [ ]:
summary = df.copy()
summary['mean_err']    = summary['mean_err'].map(lambda v: f"{v:.3e}")
summary['max_abs_err'] = summary['max_abs_err'].map(lambda v: f"{v:.3e}")
for c in ['mean_ms', 'min_ms', 'max_ms']:
    summary[c] = summary[c].map(lambda v: f"{v:.3f}")
summary['bandwidth_GBs'] = summary['bandwidth_GBs'].map(lambda v: f"{v:.1f}")
summary.to_csv('results_summary.csv', index=False)
summary
